In [1]:
# ============================================
# 1. Imports et configuration
# ============================================

from pathlib import Path
import json
import warnings

warnings.filterwarnings("ignore")

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# -------------------------------------------------
# Détection automatique de la racine du projet
# -------------------------------------------------
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
MODELS_DIR = RESULTS_DIR / "models"

# Création des dossiers s'ils n'existent pas
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("FEATURES_DIR :", FEATURES_DIR)
print("RESULTS_DIR  :", RESULTS_DIR)

PROJECT_ROOT : c:\Users\espacegamers\Desktop\Master IAII\Cours\drive-download-20251005T182208Z-1-001\S2\Traitement d_images et vision par ordinateur_\Projet\industrial-defect-detection
RAW_DIR      : c:\Users\espacegamers\Desktop\Master IAII\Cours\drive-download-20251005T182208Z-1-001\S2\Traitement d_images et vision par ordinateur_\Projet\industrial-defect-detection\data\raw
FEATURES_DIR : c:\Users\espacegamers\Desktop\Master IAII\Cours\drive-download-20251005T182208Z-1-001\S2\Traitement d_images et vision par ordinateur_\Projet\industrial-defect-detection\data\features
RESULTS_DIR  : c:\Users\espacegamers\Desktop\Master IAII\Cours\drive-download-20251005T182208Z-1-001\S2\Traitement d_images et vision par ordinateur_\Projet\industrial-defect-detection\results


In [3]:
# ============================================
# 2. Fonctions utilitaires pour construire
#    une première table de features
# ============================================

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def infer_label_from_path(image_path):
    """
    Déduit la classe à partir du chemin du fichier.
    Compatible avec des dossiers contenant 'ok', 'good',
    'defective', 'defect', etc.
    """
    p = str(image_path).lower()

    if "def_front" in p or "defective" in p or "defect" in p:
        return "Defective"
    if "ok_front" in p or "/ok/" in p or "\\ok\\" in p or "good" in p or "normal" in p:
        return "OK"

    return None


def get_all_image_paths(root_dir):
    """
    Récupère récursivement toutes les images sous root_dir.
    """
    image_paths = []
    for path in root_dir.rglob("*"):
        if path.suffix.lower() in IMAGE_EXTENSIONS:
            image_paths.append(path)
    return sorted(image_paths)


def largest_contour_stats(binary_mask):
    """
    Retourne le plus grand contour, son aire et son périmètre.
    """
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None, 0.0, 0.0

    largest = max(contours, key=cv2.contourArea)
    area = float(cv2.contourArea(largest))
    perimeter = float(cv2.arcLength(largest, True))

    return largest, area, perimeter


def build_piece_mask(gray):
    """
    Essaie de segmenter grossièrement la pièce par seuillage Otsu.
    On teste le masque normal et inversé puis on garde celui
    qui produit la zone la plus plausible.
    """
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    candidates = [th, cv2.bitwise_not(th)]
    h, w = gray.shape
    total_pixels = h * w

    best_mask = np.zeros_like(gray)
    best_area = 0.0

    kernel = np.ones((5, 5), np.uint8)

    for candidate in candidates:
        mask = cv2.morphologyEx(candidate, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        _, area, _ = largest_contour_stats(mask)
        area_ratio = area / total_pixels if total_pixels > 0 else 0

        # On évite les masques absurdes (trop petits ou toute l'image)
        if 0.05 <= area_ratio <= 0.95 and area > best_area:
            best_area = area
            best_mask = mask

    return best_mask


def extract_basic_features(image_path, hist_bins=16):
    """
    Extrait un premier ensemble de caractéristiques simples
    à partir d'une image grayscale.
    """
    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

    if gray is None:
        raise ValueError(f"Impossible de lire l'image : {image_path}")

    h, w = gray.shape
    total_pixels = h * w

    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    piece_mask = build_piece_mask(gray)

    # Pixels de la pièce si le masque existe, sinon toute l'image
    if np.count_nonzero(piece_mask) > 0:
        piece_pixels = gray[piece_mask > 0]
    else:
        piece_pixels = gray.flatten()

    # Statistiques de base
    features = {
        "mean_gray": float(np.mean(piece_pixels)),
        "std_gray": float(np.std(piece_pixels)),
        "min_gray": float(np.min(piece_pixels)),
        "max_gray": float(np.max(piece_pixels)),
        "median_gray": float(np.median(piece_pixels)),
        "q25_gray": float(np.percentile(piece_pixels, 25)),
        "q75_gray": float(np.percentile(piece_pixels, 75)),
    }

    # Histogramme normalisé
    hist = cv2.calcHist([gray], [0], None, [hist_bins], [0, 256]).flatten()
    hist = hist / hist.sum()

    for i, value in enumerate(hist):
        features[f"hist_{i:02d}"] = float(value)

    # Aire / périmètre approximatifs de la pièce
    largest_contour, largest_area, largest_perimeter = largest_contour_stats(piece_mask)
    features["piece_area"] = largest_area
    features["piece_perimeter"] = largest_perimeter
    features["piece_area_ratio"] = float(np.count_nonzero(piece_mask) / total_pixels) if total_pixels > 0 else 0.0

    # Densité de contours sur l'image
    edges = cv2.Canny(blur, 50, 150)

    if np.count_nonzero(piece_mask) > 0:
        edges = cv2.bitwise_and(edges, edges, mask=piece_mask)

    edge_density = float(np.count_nonzero(edges) / total_pixels) if total_pixels > 0 else 0.0
    features["edge_density"] = edge_density

    # Nombre de contours simples sur l'image de contours
    edge_contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid_edge_contours = [cnt for cnt in edge_contours if cv2.contourArea(cnt) >= 20]
    features["contour_count"] = int(len(valid_edge_contours))

    return features


def build_features_dataframe_from_images(raw_dir):
    """
    Construit un DataFrame de features à partir des images brutes.
    """
    image_paths = get_all_image_paths(raw_dir)

    records = []

    for idx, image_path in enumerate(image_paths, start=1):
        label = infer_label_from_path(image_path)

        # On ignore les images dont la classe n'est pas identifiable
        if label is None:
            continue

        feature_dict = extract_basic_features(image_path)
        feature_dict["image_path"] = str(image_path)
        feature_dict["label"] = label

        records.append(feature_dict)

        if idx % 200 == 0:
            print(f"{idx} images traitées...")

    df = pd.DataFrame(records)
    return df


def load_or_create_features_table():
    """
    Essaie d'abord de charger un CSV de features déjà existant.
    Sinon, construit automatiquement une première table de features.
    """
    candidate_files = [
        FEATURES_DIR / "features_dataset.csv",
        FEATURES_DIR / "features_dataset_basic.csv",
        PROCESSED_DIR / "features_dataset.csv",
    ]

    for csv_path in candidate_files:
        if csv_path.exists():
            df = pd.read_csv(csv_path)

            if "label" in df.columns:
                print(f"Features chargées depuis : {csv_path}")
                return df, csv_path

    print("Aucun fichier de features trouvé. Construction d'une table simple depuis les images...")
    df = build_features_dataframe_from_images(RAW_DIR)

    output_csv = FEATURES_DIR / "features_dataset_basic.csv"
    df.to_csv(output_csv, index=False)

    print(f"Features sauvegardées dans : {output_csv}")
    return df, output_csv

In [4]:
# ============================================
# 3. Chargement et inspection du dataset
# ============================================

df, source_path = load_or_create_features_table()

print("\nSource utilisée :", source_path)
print("Shape du dataset :", df.shape)

display(df.head())

print("\nInformations générales :")
df.info()

print("\nRépartition des classes :")
print(df["label"].value_counts())

print("\nValeurs manquantes par colonne :")
display(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nRésumé statistique des 10 premières colonnes numériques :")
display(df.select_dtypes(include=[np.number]).describe().T.head(10))

Aucun fichier de features trouvé. Construction d'une table simple depuis les images...
200 images traitées...
400 images traitées...
600 images traitées...
800 images traitées...
1000 images traitées...
1200 images traitées...
1400 images traitées...
1600 images traitées...
1800 images traitées...
2000 images traitées...
2200 images traitées...
2400 images traitées...
2600 images traitées...
2800 images traitées...
3000 images traitées...
3200 images traitées...
3400 images traitées...
3600 images traitées...
3800 images traitées...
4000 images traitées...
4200 images traitées...
4400 images traitées...
4600 images traitées...
4800 images traitées...
5000 images traitées...
5200 images traitées...
5400 images traitées...
5600 images traitées...
5800 images traitées...
6000 images traitées...
6200 images traitées...
6400 images traitées...
6600 images traitées...
6800 images traitées...
7000 images traitées...
7200 images traitées...
Features sauvegardées dans : c:\Users\espacegamers\De

,mean_gray,std_gray,min_gray,max_gray,median_gray,q25_gray,q75_gray,hist_00,hist_01,hist_02,...,hist_13,hist_14,hist_15,piece_area,piece_perimeter,piece_area_ratio,edge_density,contour_count,image_path,label
0,47.110958,33.983566,0.0,146.0,30.0,18.0,86.0,0.036800,0.072189,0.020078,...,0.021411,0.009422,0.000933,14441.0,449.788885,0.209189,0.012222,4,c:\Users\espacegamers\Desktop\Master IAII\Cour...,Defective
1,54.777424,33.706243,0.0,140.0,46.0,24.0,89.0,0.021944,0.082778,0.027767,...,0.019267,0.008144,0.001300,25858.0,918.406197,0.261833,0.018900,6,c:\Users\espacegamers\Desktop\Master IAII\Cour...,Defective
2,37.067836,28.943062,0.0,137.0,23.0,15.0,56.0,0.048844,0.069444,0.022067,...,0.023300,0.001844,0.000122,13758.0,439.788883,0.193767,0.010522,1,c:\Users\espacegamers\Desktop\Master IAII\Cour...,Defective
3,54.838243,33.677879,1.0,133.0,46.0,24.0,90.0,0.021867,0.082856,0.027911,...,0.019422,0.009311,0.001600,25836.5,944.423442,0.261367,0.017989,6,c:\Users\espacegamers\Desktop\Master IAII\Cour...,Defective
4,37.256588,30.418171,0.0,201.0,24.0,16.0,50.0,0.043478,0.079356,0.023811,...,0.016489,0.011100,0.001633,15515.5,814.825462,0.194389,0.017122,6,c:\Users\espacegamers\Desktop\Master IAII\Cour...,Defective



Informations générales :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7348 entries, 0 to 7347
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mean_gray         7348 non-null   float64
 1   std_gray          7348 non-null   float64
 2   min_gray          7348 non-null   float64
 3   max_gray          7348 non-null   float64
 4   median_gray       7348 non-null   float64
 5   q25_gray          7348 non-null   float64
 6   q75_gray          7348 non-null   float64
 7   hist_00           7348 non-null   float64
 8   hist_01           7348 non-null   float64
 9   hist_02           7348 non-null   float64
 10  hist_03           7348 non-null   float64
 11  hist_04           7348 non-null   float64
 12  hist_05           7348 non-null   float64
 13  hist_06           7348 non-null   float64
 14  hist_07           7348 non-null   float64
 15  hist_08           7348 non-null   float64
 16  hist_09         

mean_gray      0
std_gray       0
min_gray       0
max_gray       0
median_gray    0
q25_gray       0
q75_gray       0
hist_00        0
hist_01        0
hist_02        0
dtype: int64


Résumé statistique des 10 premières colonnes numériques :


,count,mean,std,min,25%,50%,75%,max
mean_gray,7348.0,44.056382,15.584880,17.388097,36.601071,41.010552,51.144476,181.094007
std_gray,7348.0,31.786480,5.279527,16.262528,28.720716,30.719136,36.881021,43.576676
min_gray,7348.0,0.421475,1.623115,0.000000,0.000000,0.000000,0.000000,24.000000
max_gray,7348.0,146.746326,32.143036,91.000000,125.000000,138.000000,156.000000,255.000000
median_gray,7348.0,31.188827,17.127063,11.000000,23.000000,27.000000,34.000000,179.000000
q25_gray,7348.0,18.510785,12.827315,7.000000,15.000000,17.000000,19.000000,156.000000
q75_gray,7348.0,69.265889,24.109651,17.000000,54.000000,62.000000,89.000000,219.000000
hist_00,7348.0,0.044970,0.016141,0.014033,0.033844,0.041489,0.054736,0.127644
hist_01,7348.0,0.068130,0.010674,0.025311,0.060611,0.067794,0.075589,0.102700
hist_02,7348.0,0.021937,0.006068,0.005233,0.018022,0.022472,0.025578,0.047400
